# Day 6: Testbenches & Simulation-Driven Development

## Pre-Class Videos (~50 minutes total)

| # | Segment | Duration | File |
|---|---------|----------|------|
| 1 | Testbench Anatomy | ~12 min | `d06_s1_testbench_anatomy.html` |
| 2 | Self-Checking Testbenches | ~15 min | `d06_s2_self_checking_testbenches.html` |
| 3 | Tasks for Organization | ~10 min | `d06_s3_tasks_for_organization.html` |
| 4 | File-Driven & Sequential Testing | ~13 min | `d06_s4_file_driven_testing.html` |

## Code Examples

| File | Description |
|------|-------------|
| `code/day06_ex01_tb_alu_template.v` | Complete self-checking ALU testbench with tasks and summary report |

## Diagrams

| File | Description |
|------|-------------|
| `diagrams/d06_sim_first_workflow.svg` | Simulate → Pass? → Synthesize → Program flowchart |
| `diagrams/d06_tb_architecture.svg` | Testbench architecture: stimulus gen → DUT → self-check → VCD |

## Key Concepts
- Testbench structure: `timescale`, clock gen, `$dumpfile/$dumpvars`, `$finish`
- Self-checking: `===`/`!==`, pass/fail counting, summary report
- Tasks for reusable stimulus/check procedures
- `$readmemh`/`$readmemb` for file-driven testing
- Sequential testbench patterns: `@(posedge clk)`, `repeat`, `wait_cycles`

## Directory Structure

```
day06_testbenches_simulation_driven_development/
├── d06_s1_testbench_anatomy.html
├── d06_s2_self_checking_testbenches.html
├── d06_s3_tasks_for_organization.html
├── d06_s4_file_driven_testing.html
├── code/
│   └── day06_ex01_tb_alu_template.v
├── diagrams/
│   ├── d06_sim_first_workflow.svg
│   └── d06_tb_architecture.svg
├── day06_quiz.md
└── day06_readme.md
```

---
## Code Examples

### `day06_ex01_tb_alu_template.v`

```verilog
// =============================================================================
// day06_ex01_tb_alu_template.v — Self-Checking ALU Testbench Template
// Day 6: Testbenches & Simulation-Driven Development
// Accelerated HDL for Digital System Design · UCF ECE
// =============================================================================
// Demonstrates: timescale, clock gen, $dumpfile/$dumpvars, structured test
// reporting with tasks, case-equality operators, and summary report.
// =============================================================================
// Build:  iverilog -o sim day06_ex01_tb_alu_template.v alu_4bit.v && vvp sim
// View:   gtkwave tb_alu.vcd
// =============================================================================

`timescale 1ns / 1ps

module tb_alu;

    // ---- DUT interface signals ----
    reg  [3:0] a, b;
    reg  [2:0] opcode;
    wire [3:0] result;
    wire       carry_out, zero;

    // ---- Instantiate DUT ----
    // (Replace with your actual ALU module name and ports)
    // alu_4bit uut (
    //     .i_a(a), .i_b(b), .i_opcode(opcode),
    //     .o_result(result), .o_carry(carry_out), .o_zero(zero)
    // );

    // ---- Placeholder for standalone demo ----
    // Simple combinational ALU for the template to compile
    reg [4:0] r_wide;
    always @(*) begin
        case (opcode)
            3'b000: r_wide = a + b;       // ADD
            3'b001: r_wide = a - b;       // SUB
            3'b010: r_wide = {1'b0, a & b}; // AND
            3'b011: r_wide = {1'b0, a | b}; // OR
            3'b100: r_wide = {1'b0, a ^ b}; // XOR
            3'b101: r_wide = {1'b0, ~a};    // NOT A
            default: r_wide = 5'd0;
        endcase
    end
    assign result    = r_wide[3:0];
    assign carry_out = r_wide[4];
    assign zero      = (result == 4'd0);

    // ---- Test counters ----
    integer test_count = 0;
    integer fail_count = 0;

    // ---- Check task ----
    task check_result;
        input [3:0] expected;
        input [3:0] actual;
        input [8*30-1:0] test_name;
    begin
        test_count = test_count + 1;
        if (actual !== expected) begin
            $display("FAIL: %0s — expected %h, got %h at time %0t",
                     test_name, expected, actual, $time);
            fail_count = fail_count + 1;
        end else begin
            $display("PASS: %0s = %h", test_name, actual);
        end
    end
    endtask

    // ---- Apply and check task (combines stimulus + verification) ----
    task apply_and_check;
        input [3:0] in_a, in_b;
        input [2:0] in_op;
        input [3:0] expected;
        input [8*30-1:0] name;
    begin
        a = in_a;
        b = in_b;
        opcode = in_op;
        #100;  // combinational settle time
        check_result(expected, result, name);
    end
    endtask

    // ---- Stimulus ----
    initial begin
        $dumpfile("tb_alu.vcd");
        $dumpvars(0, tb_alu);

        // Initialize
        a = 0; b = 0; opcode = 0;
        #50;

        $display("\n=== ALU Testbench ===\n");

        // Arithmetic tests
        apply_and_check(4'd3,  4'd5,  3'b000, 4'd8,   "ADD  3 + 5 = 8");
        apply_and_check(4'd15, 4'd1,  3'b000, 4'd0,   "ADD 15 + 1 = 0 (overflow)");
        apply_and_check(4'd7,  4'd3,  3'b001, 4'd4,   "SUB  7 - 3 = 4");
        apply_and_check(4'd0,  4'd1,  3'b001, 4'hF,   "SUB  0 - 1 = F (underflow)");

        // Logic tests
        apply_and_check(4'hA,  4'hC,  3'b010, 4'h8,   "AND  A & C = 8");
        apply_and_check(4'hA,  4'hC,  3'b011, 4'hE,   "OR   A | C = E");
        apply_and_check(4'hA,  4'h5,  3'b100, 4'hF,   "XOR  A ^ 5 = F");
        apply_and_check(4'hF,  4'h0,  3'b101, 4'h0,   "NOT  ~F = 0");

        // Edge cases
        apply_and_check(4'd0,  4'd0,  3'b000, 4'd0,   "ADD  0 + 0 = 0 (zero)");
        apply_and_check(4'hF,  4'hF,  3'b010, 4'hF,   "AND  F & F = F (identity)");

        // ---- Summary Report ----
        $display("\n=== TEST SUMMARY ===");
        $display("Tests run:    %0d", test_count);
        $display("Tests passed: %0d", test_count - fail_count);
        $display("Tests failed: %0d", fail_count);
        if (fail_count == 0)
            $display("\n*** ALL TESTS PASSED ***\n");
        else
            $display("\n*** %0d FAILURES — FIX BEFORE PROCEEDING ***\n", fail_count);

        $finish;
    end

endmodule
```

---
## Pre-Class Self-Check Quiz

**Q1:** Why are DUT inputs declared as `reg` in the testbench, but outputs as `wire`?

<details><summary>Answer</summary>
In the testbench, **you** drive the DUT's inputs (assigning them in `initial`/`always` blocks), so they must be `reg`. DUT outputs are driven by the DUT itself (a module output), so they are `wire` — you observe them, you don't drive them.
</details>

**Q2:** Why should you use `!==` instead of `!=` in testbench comparisons?

<details><summary>Answer</summary>
`!==` (case inequality) properly handles `x` and `z` values. `x !== 0` is **true** (they definitely don't match). `x != 0` is **x** (unknown) — which means your if-statement doesn't trigger and the bug goes undetected. Always use `===`/`!==` in testbenches.
</details>

**Q3:** What is the purpose of the `$dumpfile` and `$dumpvars` system tasks?

<details><summary>Answer</summary>
`$dumpfile("name.vcd")` specifies the output waveform file name. `$dumpvars(0, tb_module)` records all signal changes at all hierarchy levels below the specified module. Together they generate the VCD file that GTKWave uses for waveform visualization.
</details>

**Q4:** Write a `task` that applies two 4-bit inputs to signals `a` and `b`, waits one clock cycle, and checks that `result` matches an expected value.

<details><summary>Answer</summary>

```verilog
task apply_and_check;
    input [3:0] in_a, in_b, expected;
    input [8*20-1:0] name;
begin
    a = in_a;
    b = in_b;
    @(posedge clk);
    @(posedge clk);
    if (result !== expected)
        $display("FAIL: %0s — expected %h, got %h", name, expected, result);
    else
        $display("PASS: %0s", name);
end
endtask
```
</details>